<a href="https://colab.research.google.com/github/liupei-wq/topic/blob/main/XPS%E6%95%B8%E6%93%9A%E8%99%95%E7%90%862.0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛠️ XPS 數據整合處理中心
此單元整合了檔案讀取、自動對齊（內插）與平均化繪圖功能。

In [6]:
# @title
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import io
from scipy.interpolate import interp1d
from google.colab import files
import re

# 建立全域變數作為橋樑
shared_averaged_data = None

def parse_structured_xps(content_str):
    lines = content_str.splitlines()
    x_vals, y_vals = [], []
    mode = None
    for line in lines:
        line = line.strip()
        if not line: continue
        if 'Dimension 1 scale=' in line:
            vals = line.split('=', 1)[1].split()
            x_vals.extend([float(v) for v in vals])
            mode = 'X'
            continue
        if '[Data 1]' in line or 'Data=' in line:
            if 'Data=' in line:
                vals = line.split('=', 1)[1].split()
                y_vals.extend([float(v) for v in vals])
            mode = 'Y'
            continue
        if line.startswith('[') and mode is not None:
            if mode == 'Y': break
            mode = None
            continue
        if mode == 'X':
            vals = line.split()
            x_vals.extend([float(v) for v in vals if v.replace('.','',1).replace('E','',1).replace('+','',1).replace('-','',1).isdigit()])
        elif mode == 'Y':
            vals = line.split()
            if len(vals) >= 2: y_vals.append(float(vals[1]))
            elif len(vals) == 1: y_vals.append(float(vals[0]))
    x, y = np.array(x_vals), np.array(y_vals)
    if len(x) > 0 and len(y) > 0:
        min_len = min(len(x), len(y))
        x, y = x[:min_len], y[:min_len]
        idx = np.argsort(x)
        return x[idx], y[idx]
    raise ValueError("解析失敗")

multi_uploader = widgets.FileUpload(accept='.txt, .csv', description='上傳多個數據', multiple=True, layout={'width': '250px'})
range_slider = widgets.FloatRangeSlider(value=[0, 100], min=-1000, max=3000, step=0.01, description='能量區間:', layout={'width': '80%'}) # This line was also inside a string in the original content.
points_input = widgets.IntText(value=601, description='建議點數:', layout={'width': '200px'})
btn_align = widgets.Button(description="🚀 執行平均化", button_style='primary')
output_area = widgets.Output()

data_list = {}

def on_multi_upload(change):
    if not change['new']: return
    with output_area:
        clear_output()
        data_list.clear()
        raw_files = change['new']
        all_files = raw_files.values() if isinstance(raw_files, dict) else raw_files

        global_min_x, global_max_x = -np.inf, np.inf
        display_min, display_max = np.inf, -np.inf
        min_step = np.inf

        for file_info in all_files:
            try:
                name = file_info.get('metadata', {}).get('name', 'Unknown') if isinstance(file_info, dict) else file_info.name
                content = file_info.get('content') if isinstance(file_info, dict) else file_info.content
            except:
                name, content = getattr(file_info, 'name', 'Unknown'), getattr(file_info, 'content', b'')

            for enc in ['utf-8', 'latin1', 'big5', 'utf-16']:
                try:
                    content_str = content.decode(enc)
                    x, y = parse_structured_xps(content_str)
                    data_list[name] = (x, y)
                    global_min_x, global_max_x = max(global_min_x, x.min()), min(global_max_x, x.max())
                    display_min, display_max = min(display_min, x.min()), max(display_max, x.max())
                    if len(x) > 1:
                        steps = np.diff(x)
                        avg_step = np.abs(np.mean(steps))
                        min_step = min(min_step, avg_step)
                    print(f"✅ {name} 載入成功")
                    break
                except: continue

        if data_list:
            range_slider.min, range_slider.max = -10000, 10000
            range_slider.min, range_slider.max = display_min, display_max
            range_slider.value = [global_min_x, global_max_x]
            suggested_points = int(np.round((global_max_x - global_min_x) / min_step)) + 1
            points_input.value = suggested_points

multi_uploader.observe(on_multi_upload, names='value')

def run_multi_alignment(b):
    global shared_averaged_data
    with output_area:
        clear_output(wait=True)
        if not data_list:
            print("❌ 請先上傳數據檔案"); return

        s, e = sorted(range_slider.value)
        new_x = np.linspace(s, e, points_input.value)
        all_intensities = []

        for name, (x, y) in data_list.items():
            f_interp = interp1d(x, y, kind='linear', fill_value="extrapolate")
            all_intensities.append(f_interp(new_x))

        avg_y = np.mean(all_intensities, axis=0)

        plt.figure(figsize=(10, 6))
        plt.plot(new_x, avg_y, color='black', lw=2.5, label=f'AVERAGE ({len(all_intensities)} spectra)')
        plt.gca().invert_xaxis()
        plt.xlabel("Binding Energy (eV)"); plt.ylabel("Intensity")
        plt.title("XPS Averaged Spectrum")
        plt.legend(); plt.grid(True, alpha=0.3)
        plt.tight_layout(); plt.show()

        final_df = pd.DataFrame({'Energy_eV': new_x, 'Average_Intensity': avg_y})
        shared_averaged_data = final_df
        print("✅ 平均數據已備妥")

        # 檔名修改與匯出功能
        filename_text = widgets.Text(value='XPS_Averaged_Only.csv', description='匯出檔名:', layout={'width': '300px'})
        btn_export = widgets.Button(description="📥 匯出平均數據 CSV", button_style='success', layout={'width':'250px'})

        def on_export_clicked(b):
            fname = filename_text.value if filename_text.value.endswith('.csv') else filename_text.value + '.csv'
            final_df.to_csv(fname, index=False)
            files.download(fname)

        btn_export.on_click(on_export_clicked)
        display(widgets.HBox([filename_text, btn_export]))

btn_align.on_click(run_multi_alignment)
display(widgets.VBox([
    widgets.HTML("<h3>🛠️ XPS 多檔案平均化工具</h3>"),
    multi_uploader, range_slider,
    widgets.HBox([points_input, btn_align]),
    output_area
]))

# -----------------------------------------------------------------------------------

# -----------------------------------------------------------------------------------

# ⚖️ XPS 能量校正 (Energy Calibration)
此單元用於利用標準品（如 Au 4f7/2 = 84.0 eV）計算位移量，並修正目標數據。

In [12]:
# @title
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import io
from scipy.signal import find_peaks
from google.colab import files

# --- 核心解析函式 ---
def robust_load_xps(content_bytes):
    for enc in ['utf-8', 'latin1', 'big5', 'utf-16']:
        try:
            content_str = content_bytes.decode(enc)
            x, y = parse_structured_xps(content_str)
            return pd.DataFrame({'Energy': x, 'Intensity': y})
        except:
            continue
    try:
        df = pd.read_csv(io.BytesIO(content_bytes))
        if df.shape[1] < 2:
            df = pd.read_csv(io.BytesIO(content_bytes), sep='\\s+', engine='python', header=None)
        return df.dropna().select_dtypes(include=[np.number])
    except:
        raise ValueError("無法解析檔案")

au_uploader = widgets.FileUpload(accept='.txt, .csv', description='1. 上傳 Au 標準品', layout={'width': '200px'})
data_uploader = widgets.FileUpload(accept='.csv, .txt', description='2. 上傳待校正數據 (選填)', layout={'width': '220px'})
btn_calibrate = widgets.Button(description="🚀 執行能量校正", button_style='danger')
calib_output = widgets.Output()

calibration_info = {'offset': 0.0}

def on_au_upload(change):
    with calib_output:
        clear_output()
        file_info = list(change['new'].values())[-1]
        content = file_info['content'] if isinstance(file_info, dict) else file_info.content
        try:
            df = robust_load_xps(content)
            x, y = df.iloc[:, 0].values, df.iloc[:, 1].values
            peaks, _ = find_peaks(y, height=np.max(y)*0.5, distance=20)
            if len(peaks) > 0:
                best_peak_idx = peaks[np.argmax(y[peaks])]
                measured_e = x[best_peak_idx]
                offset = 84.0 - measured_e
                calibration_info['offset'] = offset
                print(f"✅ Au 偵測完成 | 位移量 (ΔE): {offset:+.3f} eV")
                if shared_averaged_data is not None:
                    print("🔗 偵測到上方平均化數據，校正將直接套用於該數據")
                plt.figure(figsize=(6, 3))
                plt.plot(x, y, 'g-', label='Au Spectrum')
                plt.axvline(measured_e, color='r', linestyle='--', label=f'Peak: {measured_e:.2f}')
                plt.gca().invert_xaxis()
                plt.xlabel("Binding Energy (eV)"); plt.title("Au Peak Detection"); plt.legend(); plt.grid(True, alpha=0.3); plt.show()
            else: print("❌ 無法偵測到明顯 Au 峰值")
        except Exception as e: print(f"❌ 讀取 Au 失敗: {e}")

def run_calibration(b):
    with calib_output:
        target_df = None
        if data_uploader.value:
            file_info = list(data_uploader.value.values())[-1]
            content = file_info['content'] if isinstance(file_info, dict) else file_info.content
            target_df = robust_load_xps(content)
            print("📁 使用手動上傳的樣品數據")
        elif 'shared_averaged_data' in globals() and shared_averaged_data is not None:
            target_df = shared_averaged_data.copy()
            print("🔗 使用自動接軌的平均樣品數據")

        if target_df is None:
            print("❌ 找不到樣品數據：請先在上方執行平均化，或手動上傳數據"); return

        try:
            offset = calibration_info['offset']
            energy_col = target_df.columns[0]
            target_df[energy_col] = target_df[energy_col] + offset

            plt.figure(figsize=(10, 5))
            plt.plot(target_df.iloc[:,0], target_df.iloc[:,1], 'r-', lw=2, label=f'Calibrated (Shift {offset:+.3f})')
            plt.gca().invert_xaxis()
            plt.xlabel("Binding Energy (eV)"); plt.title("Energy Calibration Result")
            plt.legend(); plt.grid(True, alpha=0.3); plt.show()

            # 檔名修改與匯出功能
            filename_text = widgets.Text(value='Calibrated_XPS_Result.csv', description='匯出檔名:', layout={'width': '300px'})
            btn_export = widgets.Button(description="📥 匯出校正後數據 CSV", button_style='success', layout={'width':'250px'})

            def on_export_clicked(b):
                fname = filename_text.value if filename_text.value.endswith('.csv') else filename_text.value + '.csv'
                target_df.to_csv(fname, index=False)
                files.download(fname)

            btn_export.on_click(on_export_clicked)
            display(widgets.HBox([filename_text, btn_export]))
            print("✅ 校正完成！請確認檔名後點擊下載")
        except Exception as e: print(f"❌ 校正失敗: {e}")

au_uploader.observe(on_au_upload, names='value')
btn_calibrate.on_click(run_calibration)

display(widgets.VBox([
    widgets.HTML("<h3>🛠️ XPS 能量校正工具 (Au 標定 - 支援自動接軌)</h3>"),
    widgets.HBox([au_uploader, data_uploader]),
    btn_calibrate, calib_output
]))

# -----------------------------------------------------------------------------------

# -----------------------------------------------------------------------------------

# 🔬 XPS 數據背景扣除 (Background Subtraction)
此單元提供 Shirley 背景扣除演算法，用於移除光譜中的非彈性散射背景，還原真實的峰值強度。

In [19]:
# @title
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import io
from google.colab import files

def shirley_background(x, y, max_iter=20):
    y = np.asarray(y)
    if len(y) < 2: return y
    bkg = np.linspace(y[0], y[-1], len(y))
    for _ in range(max_iter):
        integral = np.cumsum(y - bkg)
        if integral[-1] == 0: break
        bkg = y[-1] + (y[0] - y[-1]) * (1 - integral / integral[-1])
    return bkg

shared_subbed_data = None
uploader = widgets.FileUpload(accept='.txt, .csv', description='1. 上傳待處理數據(選填)', layout={'width': '220px'})
btn_sub = widgets.Button(description='🚀 執行背景扣除', button_style='primary')
output_panel = widgets.Output()

def process_xps_data(df):
    global shared_subbed_data
    with output_panel:
        clear_output(wait=True)
        energy_col = next((c for c in df.columns if any(k in str(c).lower() for k in ['energy', 'binding', 'ev'])), df.columns[0])
        data_cols = [c for c in df.columns if c != energy_col]
        e_min, e_max = df[energy_col].min(), df[energy_col].max()

        range_slider = widgets.FloatRangeSlider(
            value=[0, 10],
            min=0,
            max=100,
            step=0.1,
            description='選取範圍%:',
            layout={'width': '90%'},
            readout=False
        )

        range_label = widgets.Label(value="當前能量區間: 計算中...")
        filename_box = widgets.Text(value='XPS_Subbed_Result.csv', description='檔名:', layout={'width': '60%'})
        btn_export = widgets.Button(description='📥 匯出 CSV', button_style='success', layout={'width': '30%'})
        plot_area = widgets.Output()

        def redraw(_=None):
            global shared_subbed_data
            with plot_area:
                clear_output(wait=True)
                p1, p2 = range_slider.value
                e_left = e_max - (p1 / 100.0) * (e_max - e_min)
                e_right = e_max - (p2 / 100.0) * (e_max - e_min)
                s, e = sorted([e_left, e_right])
                range_label.value = f"實測選取能量區間 (左至右): {e_left:.2f} eV ~ {e_right:.2f} eV"

                fig, ax = plt.subplots(figsize=(10, 5))
                export_data = {energy_col: df[energy_col]}
                for col in data_cols:
                    x, y = df[energy_col].values, df[col].values
                    sort_idx = np.argsort(x)
                    xs, ys = x[sort_idx], y[sort_idx]
                    mask = (xs >= s) & (xs <= e)
                    if np.any(mask):
                        bkg_seg = shirley_background(xs[mask], ys[mask])
                        full_bkg = np.full_like(ys, bkg_seg[0])
                        full_bkg[mask] = bkg_seg
                        full_bkg[xs > e] = bkg_seg[-1]
                        ys_sub = ys - full_bkg
                        ax.plot(xs, ys, ':', alpha=0.4, label=f'Raw {col}')
                        ax.plot(xs, full_bkg, '--', color='gray')
                        ax.plot(xs, ys_sub, '-', lw=2, label=f'{col} (Subbed)')
                        final_y = np.zeros_like(y)
                        final_y[sort_idx] = ys_sub
                        export_data[f'{col}_Subbed'] = final_y

                shared_subbed_data = pd.DataFrame(export_data)
                ax.axvspan(s, e, color='red', alpha=0.05, label='Bkg Region')
                ax.set_title('XPS Background Subtraction (Shirley)')
                ax.invert_xaxis()
                ax.set_xlabel(f"{energy_col} (eV)")
                ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
                ax.grid(True, alpha=0.3)
                plt.tight_layout()
                plt.show()
                print("✅ 背景扣除完成，數據已同步至下方歸一化工具")

        range_slider.observe(redraw, 'value')
        btn_export.on_click(lambda b: (shared_subbed_data.to_csv(filename_box.value, index=False), files.download(filename_box.value)))
        display(widgets.VBox([
            widgets.HTML('<br><b>⚙️ 背景範圍設定 (滑桿左側=高能，右側=低能)</b>'),
            range_slider,
            range_label,
            widgets.HBox([filename_box, btn_export]),
            plot_area
        ]))
        redraw()

def run_subtraction(b):
    target_df = None
    if uploader.value:
        info = list(uploader.value.values())[-1]
        content = info['content'] if isinstance(info, dict) else info.content
        try:
            target_df = pd.read_csv(io.BytesIO(content))
        except:
            # 嘗試使用通用的讀取邏輯
            for enc in ['utf-8', 'latin1', 'big5']:
                try:
                    content_str = content.decode(enc)
                    x, y = parse_structured_xps(content_str)
                    target_df = pd.DataFrame({'Energy': x, 'Intensity': y})
                    break
                except: continue
    elif 'shared_averaged_data' in globals() and shared_averaged_data is not None:
        target_df = shared_averaged_data

    if target_df is not None: process_xps_data(target_df)
    else:
        with output_panel: print('❌ 找不到數據')

btn_sub.on_click(run_subtraction)
uploader.observe(lambda c: run_subtraction(None), names='value')
display(widgets.VBox([widgets.HTML('<h3>🔬 XPS Shirley 背景扣除工具</h3>'), widgets.HBox([uploader, btn_sub]), output_panel]))

# -----------------------------------------------------------------------------------

# -----------------------------------------------------------------------------------

# 📏 算術平均歸一化工具 (Normalization)
此單元提供階梯式歸一化功能。透過選取光譜高能量端（Post-edge）的平坦區間計算算術平均值，並將整份數據除以該數值，以達到強度標準化的目的。

In [17]:
# @title
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import io
from google.colab import files

uploader_step = widgets.FileUpload(accept='.csv, .txt', description='1. 上傳數據', layout={'width': '220px'})
btn_step_norm = widgets.Button(description='🚀 執行階梯歸一化', button_style='warning')
step_output_panel = widgets.Output()

def start_step_normalization(df):
    with step_output_panel:
        clear_output(wait=True)
        energy_col = next((c for c in df.columns if any(k in str(c).lower() for k in ['energy', 'binding', 'ev'])), df.columns[0])
        data_cols = [c for c in df.columns if c != energy_col]
        e_min, e_max = df[energy_col].min(), df[energy_col].max()

        # 使用 0-100 的百分比滑桿來模擬「左大右小」的操作感
        # 滑桿 0% 對應 e_max (左側), 100% 對應 e_min (右側)
        step_slider = widgets.FloatRangeSlider(
            value=[0, 15],
            min=0,
            max=100,
            step=0.1,
            description='選取範圍%:',
            layout={'width': '90%'},
            readout=False
        )

        range_label = widgets.Label(value="當前能量區間: 計算中...")
        filename_box = widgets.Text(value='XPS_Step_Normalized.csv', description='檔名:', layout={'width': '60%'})
        btn_export = widgets.Button(description='📥 匯出 CSV', button_style='success', layout={'width': '30%'})
        plot_area = widgets.Output()

        def redraw_step(_=None):
            with plot_area:
                clear_output(wait=True)
                p1, p2 = step_slider.value
                e_left = e_max - (p1 / 100.0) * (e_max - e_min)
                e_right = e_max - (p2 / 100.0) * (e_max - e_min)

                s, e = sorted([e_left, e_right])
                range_label.value = f"實測選取能量區間 (左至右): {e_left:.2f} eV ~ {e_right:.2f} eV"

                fig, ax = plt.subplots(figsize=(10, 5))
                export_data = {energy_col: df[energy_col]}

                for col in data_cols:
                    x, y = df[energy_col].values, df[col].values
                    mask = (x >= s) & (x <= e)
                    mean_val = np.mean(y[mask]) if np.any(mask) else np.mean(y)
                    if mean_val == 0: mean_val = 1.0
                    y_norm = y / mean_val
                    ax.plot(x, y_norm, label=f'{col} (Step Norm)')
                    export_data[f'{col}_StepNorm'] = y_norm

                final_df = pd.DataFrame(export_data)
                ax.axvspan(s, e, color='blue', alpha=0.1, label='Selected Region')
                ax.set_title('XPS Post-edge Normalization (Intuitive Slider)')
                ax.invert_xaxis()
                ax.set_xlabel(f"{energy_col} (eV)")
                ax.set_ylabel('Normalized Intensity')
                ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
                ax.grid(True, alpha=0.3)
                plt.tight_layout()
                plt.show()

                def do_export(b):
                   final_df.to_csv(filename_box.value, index=False)
                   files.download(filename_box.value)
                btn_export.on_click(do_export)

        step_slider.observe(redraw_step, 'value')
        display(widgets.VBox([
            widgets.HTML('<br><b>📏 歸一化設定 (滑桿左側=高能，右側=低能，與圖表同步)</b>'),
            step_slider,
            range_label,
            widgets.HBox([filename_box, btn_export]),
            plot_area
        ]))
        redraw_step()

def run_step_norm(b):
    target_df = None
    if uploader_step.value:
        info = list(uploader_step.value.values())[-1]
        target_df = pd.read_csv(io.BytesIO(info['content']))
    elif 'shared_subbed_data' in globals() and shared_subbed_data is not None:
        target_df = shared_subbed_data
    elif 'shared_averaged_data' in globals() and shared_averaged_data is not None:
        target_df = shared_averaged_data

    if target_df is not None: start_step_normalization(target_df)
    else:
        with step_output_panel: print('❌ 找不到數據')

btn_step_norm.on_click(run_step_norm)
display(widgets.VBox([widgets.HTML('<h3>📏 算術平均歸一化工具</h3>'), widgets.HBox([uploader_step, btn_step_norm]), step_output_panel]))

# -----------------------------------------------------------------------------------

# -----------------------------------------------------------------------------------

# 📊 數據峰值歸一化處理 (Normalization)
此單元用於將不同實驗條件下的數據進行標準化，透過指定能量範圍內的最高峰值（I/Imax = 1）來進行強度對比。

In [24]:
# @title
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import io
from google.colab import files

uploader_norm = widgets.FileUpload(accept='.csv, .txt', description='1. 手動上傳(選填)', layout={'width': '220px'})
btn_norm = widgets.Button(description='🚀 執行歸一化', button_style='success')
norm_output_panel = widgets.Output()

def start_peak_normalization(df):
    with norm_output_panel:
        clear_output(wait=True)
        energy_col = next((c for c in df.columns if any(k in str(c).lower() for k in ['energy', 'binding', 'ev'])), df.columns[0])
        data_cols = [c for c in df.columns if '_subbed' in str(c).lower() or 'intensity' in str(c).lower() or c != energy_col]
        e_min, e_max = df[energy_col].min(), df[energy_col].max()

        v1, v2 = sorted([e_min, e_max])
        peak_slider = widgets.FloatRangeSlider(
            value=[v1, v2],
            min=e_min,
            max=e_max,
            step=0.01,
            description='峰值選取區:',
            layout={'width': '90%'},
            readout_format='.2f'
        )

        filename_box = widgets.Text(value='XPS_Final_Normalized.csv', description='檔名:', layout={'width': '60%'})
        btn_export = widgets.Button(description='📥 匯出 CSV', button_style='success', layout={'width': '30%'})
        plot_area = widgets.Output()

        def redraw_norm(_=None):
            with plot_area:
                clear_output(wait=True)
                vals = sorted(peak_slider.value)
                s, e = vals[0], vals[1]
                fig, ax = plt.subplots(figsize=(10, 5))
                export_data = {energy_col: df[energy_col]}
                for col in data_cols:
                    x, y = df[energy_col].values, df[col].values
                    mask = (x >= s) & (x <= e)
                    peak_val = np.max(y[mask]) if np.any(mask) else np.max(y)
                    if peak_val == 0: peak_val = 1.0
                    y_norm = y / peak_val
                    ax.plot(x, y_norm, label=f'{col} (Norm)')
                    export_data[f'{col}_Norm'] = y_norm
                final_df = pd.DataFrame(export_data)
                ax.axvspan(s, e, color='green', alpha=0.05, label='Peak Search Range')
                ax.set_title('XPS Peak Normalization')
                ax.invert_xaxis()
                ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
                ax.grid(True, alpha=0.3)
                plt.tight_layout()
                plt.show()
                btn_export.on_click(lambda b: (final_df.to_csv(filename_box.value, index=False), files.download(filename_box.value)))

        peak_slider.observe(redraw_norm, 'value')
        display(widgets.VBox([widgets.HTML('<br><b>🎯 歸一化設定 (滑桿由低能到高能)</b>'), peak_slider, widgets.HBox([filename_box, btn_export]), plot_area]))
        redraw_norm()

def run_norm(b):
    target_df = None
    if uploader_norm.value:
        info = list(uploader_norm.value.values())[-1]
        target_df = pd.read_csv(io.BytesIO(info['content']))
    elif 'shared_subbed_data' in globals() and shared_subbed_data is not None:
        target_df = shared_subbed_data
    if target_df is not None: start_peak_normalization(target_df)
    else:
        with norm_output_panel: print('❌ 找不到數據')

btn_norm.on_click(run_norm)
display(widgets.VBox([widgets.HTML('<h3>📊 XPS 峰值歸一化工具</h3>'), widgets.HBox([uploader_norm, btn_norm]), norm_output_panel]))

# -----------------------------------------------------------------------------------

# -----------------------------------------------------------------------------------

# 📐 數據面積歸一化工具 (Area Normalization)

In [25]:
# @title
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import io
from google.colab import files

# UI 元件建立
uploader_area = widgets.FileUpload(accept='.csv, .txt', description='1. 上傳數據(選填)', layout={'width': '220px'})
btn_area_norm = widgets.Button(description='🚀 執行面積歸一化', button_style='info')
area_output_panel = widgets.Output()

def start_area_normalization(df):
    with area_output_panel:
        clear_output(wait=True)

        # 偵測能量與強度欄位
        energy_col = next((c for c in df.columns if any(k in str(c).lower() for k in ['energy', 'binding', 'ev'])), df.columns[0])
        data_cols = [c for c in df.columns if c != energy_col]

        filename_box = widgets.Text(value='XPS_Area_Normalized.csv', description='檔名:', layout={'width': '60%'})
        btn_export = widgets.Button(description='📥 匯出 CSV', button_style='success', layout={'width': '30%'})
        plot_area = widgets.Output()

        def perform_norm():
            with plot_area:
                clear_output(wait=True)
                fig, ax = plt.subplots(figsize=(10, 5))
                export_data = {energy_col: df[energy_col]}

                for col in data_cols:
                    x = df[energy_col].values
                    y = df[col].values

                    # 排序數據以確保積分正確
                    sort_idx = np.argsort(x)
                    xs, ys = x[sort_idx], y[sort_idx]

                    # 使用 trapezoid 計算總面積
                    try:
                        total_area = np.abs(np.trapezoid(ys, xs))
                    except AttributeError:
                        total_area = np.abs(np.trapz(ys, xs))

                    if total_area == 0: total_area = 1.0
                    y_norm = y / total_area

                    ax.plot(x, y_norm, label=f'{col} (Area Norm, A={total_area:.2e})')
                    export_data[f'{col}_AreaNorm'] = y_norm

                final_df = pd.DataFrame(export_data)
                ax.set_title('XPS Area Normalization (Total Area = 1)')
                ax.invert_xaxis()
                ax.set_xlabel(f"{energy_col} (eV)")
                ax.set_ylabel('Normalized Intensity (Area=1)')
                ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
                ax.grid(True, alpha=0.3)
                plt.tight_layout()
                plt.show()

                btn_export.on_click(lambda b: (final_df.to_csv(filename_box.value, index=False), files.download(filename_box.value)))

        display(widgets.VBox([
            widgets.HTML('<p>此工具將計算整條曲線面積並將其歸一化為 1</p>'),
            widgets.HBox([filename_box, btn_export]),
            plot_area
        ]))
        perform_norm()

def run_area_norm(b):
    target_df = None
    # 優先順序：手動上傳 > 背景扣除數據 (shared_subbed_data) > 平均化數據 (shared_averaged_data)
    if uploader_area.value:
        info = list(uploader_area.value.values())[-1]
        target_df = pd.read_csv(io.BytesIO(info['content']))
    elif 'shared_subbed_data' in globals() and shared_subbed_data is not None:
        target_df = shared_subbed_data
        with area_output_panel: print("🔗 自動讀取：背景扣除後的數據")
    elif 'shared_averaged_data' in globals() and shared_averaged_data is not None:
        target_df = shared_averaged_data
        with area_output_panel: print("🔗 自動讀取：平均化後的原始數據")

    if target_df is not None:
        start_area_normalization(target_df)
    else:
        with area_output_panel: print('❌ 找不到數據')

btn_area_norm.on_click(run_area_norm)
display(widgets.VBox([
    widgets.HTML('<h3>━━ 數據面積歸一化入口 ━━</h3>'),
    widgets.HBox([uploader_area, btn_area_norm]),
    area_output_panel
]))

# -----------------------------------------------------------------------------------

# -----------------------------------------------------------------------------------

# 🎯 曲線擬合 (Curve Fitting)
此單元提供高階的分峰擬合功能，包含 Pseudo-Voigt 函數模型與自動峰值偵測。

In [ ]:
# @title
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from scipy.signal import savgol_filter, find_peaks
from scipy.optimize import curve_fit
from IPython.display import display, clear_output
from google.colab import files
import io

# --- 1. 數學模型定義 ---
def gaussian(x, amp, cen, fwhm):
    sig = fwhm / (2 * np.sqrt(2 * np.log(2)))
    return amp * np.exp(-(x - cen)**2 / (2 * sig**2))

def lorentzian(x, amp, cen, fwhm):
    return amp * (fwhm**2 / ((x - cen)**2 + fwhm**2))

def pseudo_voigt(x, amp, cen, fwhm, eta):
    """Pseudo-Voigt 函數: (1-eta)*Gaussian + eta*Lorentzian"""
    return (1 - eta) * gaussian(x, amp, cen, fwhm) + eta * lorentzian(x, amp, cen, fwhm)

def shirley_background(x, y, max_iter=20):
    """Shirley 背景計算"""
    y = np.array(y)
    if len(y) < 2: return y
    bkg = np.linspace(y[0], y[-1], len(y))
    for _ in range(max_iter):
        integral = np.cumsum(y - bkg)
        if integral[-1] == 0: break
        bkg = y[-1] + (y[0] - y[-1]) * (1 - integral / integral[-1])
    return bkg

# --- 2. 擬合引擎 ---
class XPSPeakFitter:
    def __init__(self):
        self.output = widgets.Output()
        self.data = None
        self.last_result_df = None

    def combined_model(self, x, *params):
        y_total = np.zeros_like(x)
        for i in range(0, len(params), 4):
            amp, cen, fwhm, eta = params[i:i+4]
            y_total += pseudo_voigt(x, amp, cen, fwhm, eta)
        return y_total

    def run_fitting(self, df, num_peaks=2):
        with self.output:
            clear_output(wait=True)
            if df is None:
                print("❌ 請先載入數據"); return

            x_raw, y_raw = df.iloc[:,0].values, df.iloc[:,1].values
            idx = np.argsort(x_raw)
            x, y = x_raw[idx], y_raw[idx]

            y_smooth = savgol_filter(y, min(11, len(y)-1 if len(y)%2!=0 else len(y)-2), 3)
            bkg = shirley_background(x, y_smooth)
            y_sub = y_smooth - bkg
            y_sub[y_sub < 0] = 0

            peaks, _ = find_peaks(y_sub, height=np.max(y_sub)*0.1, distance=10)
            if len(peaks) < num_peaks:
                 peaks = np.argsort(y_sub)[-num_peaks:]
            else:
                 peaks = peaks[np.argsort(y_sub[peaks])[-num_peaks:]]
            peaks = np.sort(peaks)

            p0, bounds_l, bounds_h = [], [], []
            for p in peaks:
                p0 += [y_sub[p], x[p], 1.2, 0.5]
                bounds_l += [0, x[p]-1.5, 0.4, 0]
                bounds_h += [y_sub[p]*1.5, x[p]+1.5, 4.0, 1]

            try:
                popt, _ = curve_fit(self.combined_model, x, y_sub, p0=p0, bounds=(bounds_l, bounds_h))

                plt.figure(figsize=(10, 6))
                plt.plot(x, y_sub, 'ko', markersize=3, alpha=0.3, label='Data (BG Subbed)')
                plt.plot(x, self.combined_model(x, *popt), 'r-', lw=2, label='Total Fit')

                for i in range(0, len(popt), 4):
                    peak_y = pseudo_voigt(x, *popt[i:i+4])
                    plt.fill_between(x, peak_y, alpha=0.3, label=f'Peak {popt[i+1]:.2f} eV')

                plt.gca().invert_xaxis()
                plt.xlabel("Binding Energy (eV)"); plt.ylabel("Intensity")
                plt.title(f"XPS Fitting: {num_peaks} Peaks")
                plt.legend(); plt.grid(True); plt.show()

                self.last_result_df = pd.DataFrame(np.array(popt).reshape(-1, 4),
                                      columns=['Amplitude', 'Position (eV)', 'FWHM', 'LG_Ratio'])
                display(self.last_result_df)
                btn_download_fit.disabled = False
                filename_fit.disabled = False
            except Exception as e: print(f"❌ 擬合失敗: {e}")

# --- 3. UI 介面 ---
fitter = XPSPeakFitter()
uploader = widgets.FileUpload(accept='.csv, .txt', description="1. 上傳數據", layout={'width': '200px'})
peak_slider = widgets.IntSlider(value=2, min=1, max=8, description="2. 分峰數量")
btn_fit = widgets.Button(description="3. 執行擬合", button_style='danger')
filename_fit = widgets.Text(value='Fitting_Result.csv', description='檔名:', layout={'width': '200px'}, disabled=True)
btn_download_fit = widgets.Button(description="📥 匯出擬合結果", button_style='success', disabled=True)

def load_data(change):
    if not uploader.value: return
    file = list(change['new'].values())[0]
    content = io.BytesIO(file['content'])
    try:
        df = pd.read_csv(content)
        if df.shape[1] < 2:
            content.seek(0)
            df = pd.read_csv(content, sep=r'\\s+', header=None, engine='python')
    except:
        content.seek(0)
        df = pd.read_csv(content, sep=r'\\s+', header=None, comment='#')
    fitter.data = df.dropna().select_dtypes(include=[np.number])
    with fitter.output:
        clear_output(); print(f"✅ 已載入檔案: {file['metadata']['name']}")

def download_fit_result(b):
    if fitter.last_result_df is not None:
        fname = filename_fit.value if filename_fit.value.endswith('.csv') else filename_fit.value + '.csv'
        fitter.last_result_df.to_csv(fname, index=False)
        files.download(fname)

uploader.observe(load_data, names='value')
btn_fit.on_click(lambda b: fitter.run_fitting(fitter.data, peak_slider.value))
btn_download_fit.on_click(download_fit_result)

display(widgets.VBox([
    widgets.HTML("<h3>🔬 XPS 高階曲線擬合與匯出工具</h3>"),
    widgets.HBox([uploader, peak_slider, btn_fit]),
    widgets.HBox([filename_fit, btn_download_fit]),
    fitter.output
]))

# -----------------------------------------------------------------------------------

# -----------------------------------------------------------------------------------

# 📉 XAS 背景扣除與訊號提取 (Background Removal)
此單元專用於 XAS 數據處理，透過互動式高斯模型扣除背景，提取精準的邊緣特徵訊號。

In [9]:
# @title
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import re
import io

# --- 修正版 XAS 數據讀取器 ---
xas_uploader = widgets.FileUpload(accept='.txt, .csv', description='1. Upload Data', layout={'width': '250px'})
e_start = widgets.FloatText(value=0, description='Energy Start:', layout={'width': '200px'})
e_end = widgets.FloatText(value=100, description='Energy End:', layout={'width': '200px'})
btn_generate = widgets.Button(description='2. Generate Plot', button_style='info')
output_load = widgets.Output()

df_xas_data = None

# 新增兩個下拉選單用於選擇TFY和TEY欄位
tfy_selector = widgets.Dropdown(description='TFY Column:', options=[], layout={'width': '200px'})
tey_selector = widgets.Dropdown(description='TEY Column:', options=[], layout={'width': '200px'})
btn_confirm_cols = widgets.Button(description='Confirm Columns', button_style='success')

def plot_data(b):
    global df_xas_data
    with output_load:
        if df_xas_data is None: print("❌ 無數據。"); return
        plt.figure(figsize=(10, 5))
        # 確保只繪製數值欄位，並排除Energy
        for col in [c for c in df_xas_data.columns if c != 'Energy' and pd.api.types.is_numeric_dtype(df_xas_data[c])]:
            plt.plot(df_xas_data['Energy'], df_xas_data[col], label=col)
        plt.xlabel('Energy (eV)'); plt.ylabel('Intensity'); plt.legend(); plt.grid(True, alpha=0.3); plt.show()

def load_xas_data(change):
    global df_xas_data
    with output_load:
        clear_output()
        if not xas_uploader.value: return
        file_info = list(xas_uploader.value.values())[0]
        content_bytes = file_info['content']
        try:
            raw_text = content_bytes.decode('utf-8', errors='replace')
            lines = raw_text.splitlines()
            numeric_rows = []
            for line in lines:
                parts = re.split(r'\s+', line.strip())
                if not parts or parts == ['']: continue
                try:
                    vals = [float(p) for p in parts if p]
                    if len(vals) >= 2:
                        numeric_rows.append(vals)
                except ValueError: continue
            if not numeric_rows:
                print("❌ 找不到數值。")
                return
            df_temp = pd.DataFrame(numeric_rows)
            # 不在這裡直接命名，先保留原始欄位名稱或預設名稱
            # 如果有超過10個欄位，可能檔案格式不同，只取前N個
            if df_temp.shape[1] > 10: # 假設超過10個欄位是異常，只取前3個
                df_temp = df_temp.iloc[:, :3]
            # 使用預設名稱或如果第一行有字串嘗試作為header
            if not df_temp.empty and isinstance(lines[0], str) and any(char.isalpha() for char in lines[0]):
                try:
                    header_candidates = re.split(r'\s+', lines[0].strip())
                    # 篩選掉空字串或數字開頭的字串，只取看起來像header的部分
                    valid_headers = [h for h in header_candidates if h and not h[0].isdigit()]
                    if len(valid_headers) == df_temp.shape[1]:
                        df_temp.columns = valid_headers
                    else:
                        df_temp.columns = [f'Col{i+1}' for i in range(df_temp.shape[1])]
                except:
                    df_temp.columns = [f'Col{i+1}' for i in range(df_temp.shape[1])]
            else:
                df_temp.columns = [f'Col{i+1}' for i in range(df_temp.shape[1])]

            # 確保Energy欄位存在並排序
            energy_col_name = None
            for col in df_temp.columns:
                if 'energy' in str(col).lower() or 'ev' in str(col).lower() or 'col1' in str(col).lower():
                    energy_col_name = col
                    break
            if energy_col_name is None: # Fallback if no energy-like column found
                energy_col_name = df_temp.columns[0]

            df_xas_data = df_temp.rename(columns={energy_col_name: 'Energy'}).sort_values(by='Energy')

            available_cols = [c for c in df_xas_data.columns if c != 'Energy']
            if len(available_cols) >= 2: # 假設通常會有TFY和TEY
                tfy_selector.options = available_cols
                tey_selector.options = available_cols
                tfy_selector.value = available_cols[0] # 預設選第一個
                tey_selector.value = available_cols[1] # 預設選第二個
            elif len(available_cols) == 1:
                tfy_selector.options = available_cols
                tey_selector.options = available_cols
                tfy_selector.value = available_cols[0]
                tey_selector.value = available_cols[0]
            else:
                tfy_selector.options = ['None']
                tey_selector.options = ['None']
                tfy_selector.value = 'None'
                tey_selector.value = 'None'

            e_start.value = df_xas_data['Energy'].min()
            e_end.value = df_xas_data['Energy'].max()
            print(f"✅ 成功載入: {file_info['metadata']['name']}")
            display(widgets.HBox([tfy_selector, tey_selector, btn_confirm_cols]))
        except Exception as e: print(f"❌ 解析失敗: {e}")

def confirm_cols(b):
    global df_xas_data
    with output_load:
        clear_output(wait=True)
        if df_xas_data is None:
            print("❌ 請先載入數據。")
            return
        if tfy_selector.value == 'None' and tey_selector.value == 'None':
            print("❌ 請選擇有效的欄位。")
            return

        # 重命名選定的欄位
        new_df = pd.DataFrame({'Energy': df_xas_data['Energy']})
        if tfy_selector.value != 'None':
            new_df['TFY_avg'] = df_xas_data[tfy_selector.value]
        if tey_selector.value != 'None' and tey_selector.value != tfy_selector.value: # 避免同名覆蓋
            new_df['TEY_avg'] = df_xas_data[tey_selector.value]
        elif tey_selector.value != 'None' and tey_selector.value == tfy_selector.value:
            print("⚠️ TFY和TEY欄位相同，請確認是否為單一信號處理。")
            new_df['TEY_avg'] = df_xas_data[tey_selector.value]

        df_xas_data = new_df
        print(f"✅ 欄位已確認：TFY_avg -> {tfy_selector.value}, TEY_avg -> {tey_selector.value}")
        display(btn_generate) # 顯示生成圖表的按鈕

xas_uploader.observe(load_xas_data, names='value')
btn_generate.on_click(plot_data)
btn_confirm_cols.on_click(confirm_cols)

display(widgets.VBox([
    widgets.HTML('<h3>XAS Data Processor</h3>'),
    xas_uploader,
    widgets.HBox([e_start, e_end]),
    output_load # 將 output_load 放在這裡，以便顯示確認欄位的介面
]))


In [10]:
# @title
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
import pandas as pd

# --- Gaussian Function ---
def gaussian(x, amp, cen, fwhm):
    sig = fwhm / (2 * np.sqrt(2 * np.log(2)))
    return amp * np.exp(-(x - cen)**2 / (2 * sig**2))

# --- UI Elements ---
column_select = widgets.Dropdown(options=['TFY_avg', 'TEY_avg'], description='Target Data:', layout={'width': '250px'})

g_amp = widgets.FloatSlider(value=0.5, min=0, max=2.0, step=0.001, description='Amplitude:', layout={'width': '400px'}, readout_format='.3f')
g_cen = widgets.FloatSlider(value=460, min=400, max=550, step=0.01, description='Center (eV):', layout={'width': '400px'})
g_fwhm = widgets.FloatSlider(value=1.0, min=0.1, max=10.0, step=0.01, description='FWHM:', layout={'width': '400px'})

btn_export_xas = widgets.Button(description='📥 Export CSV', button_style='success')
output_sub = widgets.Output()

df_final_xas = None

def redraw_subtraction(_=None):
    global df_final_xas
    with output_sub:
        clear_output(wait=True)
        if 'df_xas_data' not in globals() or df_xas_data is None:
            print("❌ Please upload and confirm data columns in the processor above first.")
            return
        target_col = column_select.value
        if target_col not in df_xas_data.columns:
            print(f"❌ 選擇的欄位 '{target_col}' 不存在。")
            return
        x = df_xas_data['Energy'].values
        y_raw = df_xas_data[target_col].values
        if g_cen.min == 400 and g_cen.max == 550 and g_cen.value == 460:
            g_cen.min, g_cen.max = x.min(), x.max()
            g_cen.value = (x.min() + x.max()) / 2
            g_amp.max = np.max(y_raw) * 1.5
            g_amp.value = np.max(y_raw) * 0.2
        y_gauss = gaussian(x, g_amp.value, g_cen.value, g_fwhm.value)
        y_subbed = y_raw - y_gauss
        df_final_xas = pd.DataFrame({'Energy_eV': x, f'Original_{target_col}': y_raw, 'Gaussian_Bkg': y_gauss, 'Subtracted_Intensity': y_subbed})
        plt.figure(figsize=(12, 6))
        plt.plot(x, y_raw, 'k--', alpha=0.3, label=f'Original {target_col}')
        plt.plot(x, y_gauss, 'r-', lw=2, label='Estimated Background')
        plt.fill_between(x, y_gauss, color='red', alpha=0.1)
        plt.plot(x, y_subbed, 'b-', lw=2.5, label='Extracted Signal')
        plt.xlabel('Energy (eV)'); plt.ylabel('Intensity'); plt.title(f'Signal Extraction: {target_col}'); plt.legend(); plt.grid(True, alpha=0.2); plt.show()

def export_xas(b):
    if df_final_xas is not None:
        df_final_xas.to_csv("XAS_Extracted_Result.csv", index=False)
        files.download("XAS_Extracted_Result.csv")

for w in [g_amp, g_cen, g_fwhm]: w.observe(redraw_subtraction, names='value')
def update_column_select_options(_=None):
    if 'df_xas_data' in globals() and df_xas_data is not None:
        current_options = [c for c in df_xas_data.columns if c != 'Energy']
        if current_options:
            column_select.options = current_options
            column_select.value = current_options[0]
            redraw_subtraction()
update_column_select_options()
column_select.observe(redraw_subtraction, names='value')
btn_export_xas.on_click(export_xas)
display(widgets.VBox([widgets.HTML('<h3>📉 XAS Background Removal & Signal Extraction</h3>'), column_select, g_amp, g_cen, g_fwhm, btn_export_xas, output_sub]))